# Demo · Detección de minería artesanal con un Vision Transformer (DeiT-Small)

Este notebook carga el mejor modelo del proyecto (DeiT-Small entrenado con BCE ponderada, semilla 0),
predice las 17 etiquetas de algunas imágenes satelitales y muestra con **Grad-CAM** en qué zonas se fija
el modelo para decir "minería artesanal".

**Antes de empezar:** `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4`
(también funciona en CPU, solo que más lento). Ejecuta las celdas en orden.

In [ ]:
# 1. Instalamos timm (la misma versión del entrenamiento) y gdown para descargar los pesos desde Google Drive
!pip -q install timm==1.0.30 gdown

In [ ]:
# 2. Clonamos el repositorio con el código, los umbrales y las imágenes de ejemplo
REPO = 'https://github.com/csanchez-GH/mia203-mineria-amazonia.git'   # ← reemplazar por la URL real
!git clone -q {REPO} repo
%cd repo

In [ ]:
# 3. Descargamos los pesos del modelo desde Google Drive (no están en GitHub porque pesan unos 90 MB)
import os
ID_PESOS_VIT = '1KWGJgjA9lJpNlsOs-p3mnqlmVvXD0KNj'   # ← id de vit_bce_pond_s0.pt en Google Drive
RUTA_PESOS = 'weights/vit_bce_pond_s0.pt'
os.makedirs('weights', exist_ok=True)
if not os.path.exists(RUTA_PESOS):
    import gdown
    gdown.download(id=ID_PESOS_VIT, output=RUTA_PESOS, quiet=False)
print('Pesos listos:', os.path.getsize(RUTA_PESOS) / 1e6, 'MB')

In [ ]:
# 4. Cargamos el modelo, la lista de etiquetas y los umbrales que elegimos en validación
import sys, json, glob
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from PIL import Image
sys.path.insert(0, '.')
from src.modelos import crear_modelo
from src.datos import preparar_imagen
from src.interpretabilidad import GradCAM

config = json.load(open('configs/config_global.json'))
ETIQUETAS, TAM = config['ETIQUETAS'], config.get('TAM', 160)
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'

# Creamos la misma arquitectura (sin descargar pesos de ImageNet) y le cargamos nuestros pesos entrenados
modelo = crear_modelo('vit', len(ETIQUETAS), TAM, preentrenado=False)
modelo.load_state_dict(torch.load(RUTA_PESOS, map_location='cpu'))
modelo = modelo.to(dispositivo).eval()

# Umbral de decisión de cada etiqueta, elegido en validación durante el proyecto
umbrales = pd.read_csv('results/tables/por_etiqueta_vit_bce_pond_s0.csv', index_col=0).loc[ETIQUETAS, 'umbral'].values
print(f'Modelo cargado en {dispositivo} | {sum(p.numel() for p in modelo.parameters()) / 1e6:.1f} M parámetros')

In [ ]:
# 5. Elegimos las imágenes: las de la carpeta ejemplos/ del repositorio o, si no hay, las que subamos
rutas = sorted(glob.glob('ejemplos/*.jpg'))
if not rutas:
    from google.colab import files
    print('No hay imágenes en ejemplos/. Sube una o más imágenes satelitales (JPG):')
    rutas = list(files.upload().keys())
print(len(rutas), 'imágenes:', [os.path.basename(r) for r in rutas])

In [ ]:
# 6. Predecimos las etiquetas y calculamos el Grad-CAM de "minería artesanal" para cada imagen
def cargar(ruta):
    # Mismo preprocesamiento que en el entrenamiento: RGB y reducción a 160 x 160 píxeles
    with Image.open(ruta) as im:
        return np.asarray(im.convert('RGB').resize((TAM, TAM), Image.BILINEAR))

cam = GradCAM(modelo, modelo.blocks[-1].norm1, es_vit=True, tam=TAM)
j_mina = ETIQUETAS.index('artisinal_mine')

fig, axes = plt.subplots(len(rutas), 2, figsize=(7, 3.3 * len(rutas)))
axes = np.atleast_2d(axes)
for fila, ruta in enumerate(rutas):
    img = cargar(ruta)
    x = preparar_imagen(img).unsqueeze(0).to(dispositivo)
    mapa, p_mina = cam(x, j_mina)                                   # Grad-CAM y probabilidad de mina
    with torch.no_grad():
        p = torch.sigmoid(modelo(x)).cpu().numpy()[0]              # probabilidad de cada etiqueta
    predichas = [f'{e} ({p[k]:.2f})' for k, e in enumerate(ETIQUETAS) if p[k] >= umbrales[k]]
    hay_mina = p[j_mina] >= umbrales[j_mina]
    print(f'{os.path.basename(ruta)} → {", ".join(predichas)}')

    axes[fila, 0].imshow(img)
    axes[fila, 0].set_title(os.path.basename(ruta), fontsize=8)
    axes[fila, 1].imshow(img); axes[fila, 1].imshow(mapa, cmap='jet', alpha=0.45)
    axes[fila, 1].set_title(f'Grad-CAM minería | p = {p_mina:.2f} → {"MINA" if hay_mina else "sin mina"}', fontsize=8)
    for a in axes[fila]:
        a.axis('off')
plt.tight_layout(); plt.show()

**Cómo leer el resultado.** Las zonas rojas del Grad-CAM son las que más hicieron subir la probabilidad
de "minería artesanal". En el proyecto encontramos que, en los aciertos, el modelo se fija en el suelo
desnudo y las pozas de relave, y que sus falsas alarmas suelen venir de **bancos de arena y sedimento
de los ríos**, que se parecen a los relaves. Por eso este modelo debe usarse como apoyo para priorizar
revisiones, siempre con revisión humana, y no como prueba de una actividad ilegal.